In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Define the model name to load the pre-trained model
model_name = "Qwen/Qwen2.5-Coder-0.5B-instruct"

# Load the pre-trained model for causal language modeling
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,  # Intel Macs only support float32 on CPU for PyTorch
    device_map=None  # Prevent automatic selection of devices (like GPU/CPU)
).to("cpu")  # Explicitly assign the model to the CPU

# Load the tokenizer corresponding to the pre-trained model
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define the prompt to send to the model
prompt = "write a quick sort algorithm."

# Define the conversation structure with system and user messages
# The messages list is structured to simulate a chat conversation
# The system message is the introduction of the assistant
# The user message is the prompt provided to the assistant
# The assistant will generate a response based on the user's prompt
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}  # The user's input prompt
]


# Tokenize the messages into the format required by the model (but without tokenizing the content itself yet)
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,  # Don't tokenize at this step, just format the messages
    # Ensure the model receives the generation prompt for context
    add_generation_prompt=True
)

# Convert the formatted text into token IDs and ensure the tensors are on the CPU
model_inputs = tokenizer([text], return_tensors="pt").to(
    "cpu")  # Use PyTorch tensors and move to CPU

# Generate model output based on the provided inputs, restricting the output to a maximum of 512 new tokens
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512  # Limit the number of tokens generated to 512
)

# Remove the original input tokens from the generated output tokens to retain only the newly generated text
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

# Decode the generated token IDs back into human-readable text, skipping special tokens like <EOS>
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

# Output the final response generated by the model
print(response)

Sure! Here's a simple implementation of the Quick Sort algorithm in Python:

```python
def quick_sort(arr):
    if len(arr) <= 1:
        return arr

    pivot = arr[len(arr) // 2]
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]

    return quick_sort(left) + middle + quick_sort(right)

# Example usage:
arr = [3, 6, 8, 10, 1, 2, 5]
sorted_arr = quick_sort(arr)
print(sorted_arr)
```

This implementation works by selecting a pivot element from the array and partitioning the other elements into two sub-arrays: one containing all elements less than the pivot and another containing all elements greater than or equal to the pivot. It then recursively applies the same process to these sub-arrays until the base case is reached (i.e., the sub-array has one or zero elements).


Why Use pipeline?

    Simplifies model loading (no need to manually handle tokenization or generation).
    Handles batching and device management automatically.
    Easy to extend for multiple iterations.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import requests
import json
import re

# Initialize model and tokenizer
model_name = "Qwen/Qwen2.5-Coder-3B-instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,  # Intel Macs only support float32 on CPU for PyTorch
    device_map=None  # Prevent automatic selection of devices (like GPU/CPU)
).to("cpu")  # Explicitly assign the model to the CPU

tokenizer = AutoTokenizer.from_pretrained(model_name)


def fetch_webpage_content(url):
    """Fetch webpage content with error handling"""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        print(f"Fetched URL: {url}")
        print(f"response.text: {response.text}")
        return response.text
    except requests.RequestException as e:
        print(f"Error fetching URL: {e}")
        return None


def generate_response(messages):
    """Generate response using the model with chat template"""
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=1024,
        temperature=0.7
    )

    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


def parse_json(json_str):
    """Parse and validate JSON output with error recovery"""
    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        # Try to extract JSON from markdown code blocks
        json_match = re.search(r'```json\n(.*?)\n```', json_str, re.DOTALL)
        if json_match:
            try:
                return json.loads(json_match.group(1))
            except json.JSONDecodeError:
                pass
        # Try to find the first valid JSON structure
        json_match = re.search(r'(\{.*\}|\[.*\])', json_str, re.DOTALL)
        if json_match:
            try:
                return json.loads(json_match.group(0))
            except json.JSONDecodeError:
                pass
        return None


def news_pipeline(url, prompt):
    """Main pipeline to process news from URL to structured JSON"""
    # Step 1: Fetch webpage content
    content = fetch_webpage_content(url)
    if not content:
        return {"error": "Failed to fetch webpage content"}

    # Step 2: Extract news list
    extraction_messages = [
        {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
        # Truncate to avoid token limits
        {"role": "user",
            "content": f"{prompt}\n\nWebsite Content:\n{content[:15000]}"}
    ]
    news_list = generate_response(extraction_messages)

    print(f"news_list: {news_list}")

    # Step 3: Convert to JSON
    json_prompt = """Convert this news list into a JSON array of objects with:
    - title (string)
    - summary (string)
    - date (string in YYYY-MM-DD format)
    - url (string if available)
    
    News List:
    """
    conversion_messages = [
        {"role": "system", "content": "You are a JSON formatting assistant. Return only valid JSON."},
        {"role": "user", "content": f"{json_prompt}\n{news_list}"}
    ]
    json_output = generate_response(conversion_messages)

    # Step 4: Parse and validate JSON
    parsed = parse_json(json_output)
    return parsed if parsed else {"error": "Invalid JSON format", "raw_output": json_output}


# Example usage
if __name__ == "__main__":
    result = news_pipeline(
        url="https://www.bbc.com/news",
        prompt="List the top 5 current news headlines with their summaries"
    )
    print(json.dumps(result, indent=2))

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[
  {
    "title": "UK Economy Faces Challenges Amidst Brexit Uncertainty",
    "summary": "The UK economy is facing significant challenges due to ongoing Brexit negotiations and economic uncertainty.",
    "date": "YYYY-MM-DD",
    "url": "https://www.bbc.com/news/economy-58784505"
  },
  {
    "title": "Worldwide Coronavirus Outbreak Continues to Spread",
    "summary": "The global coronavirus outbreak continues to spread rapidly, with cases in multiple countries increasing daily.",
    "date": "YYYY-MM-DD",
    "url": "https://www.bbc.com/news/world-coronavirus-outbreak-58784497"
  },
  {
    "title": "US Election Results Announced: Biden Wins Presidency",
    "summary": "The US election results have been announced, with Joe Biden defeating Donald Trump to become the next President.",
    "date": "YYYY-MM-DD",
    "url": "https://www.bbc.com/news/election-us-58784495"
  },
  {
    "title": "Climate Change Impact on Global Food Supply",
    "summary": "Climate change is having a prof